# Experiment 001: All Four Models with Class 1 Fairness Analysis

### Experiment Overview
This notebook reruns Experiment 001 using the raw cleaned dataset. It trains four models (Logistic Regression, Random Forest, XGBoost, and MLP Neural Network) and evaluates both overall model performance and Class 1 fairness across race, gender, and age.

### Class Label Explanation
| Class | Meaning | Importance |
|---|---|---|
| Class 0 | Not readmitted within 30 days | Majority class |
| Class 1 | Readmitted within 30 days | Clinically important class |

**Clinical Focus Note:**  
FairCare focuses on Class 1 because Class 1 represents 30-day readmission. The main fairness question is: **does the model detect readmitted patients equally across race, gender, and age groups?**

Four Class 1 fairness matrices are calculated for each model:
1. **Performance Fairness Matrix** — Class 1 recall, precision, F1
2. **Error Fairness Matrix** — Class 1 missed patients using FN and FNR
3. **Calibration Fairness Matrix** — predicted Class 1 risk vs actual Class 1 readmission rate
4. **SHAP Explanation Fairness Matrix** — feature influence for Class 1 readmission risk


# SECTION 2: Load Cleaned Data

We load `clean_train_dataset.csv` and `clean_test_dataset.csv`.
We display their shapes, target class distribution, and available demographic columns (`race`, `gender`, `age`).

*Note: The data is already cleaned. We do not perform any additional cleaning.*


In [1]:
import pandas as pd
import numpy as np

# Load the raw cleaned datasets
train_df = pd.read_csv('clean_train_dataset.csv')
test_df = pd.read_csv('clean_test_dataset.csv')

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

print("\nTarget ('readmitted') distribution in training set:")
print(train_df['readmitted'].value_counts())
print(train_df['readmitted'].value_counts(normalize=True))

print("\nTarget ('readmitted') distribution in test set:")
print(test_df['readmitted'].value_counts())
print(test_df['readmitted'].value_counts(normalize=True))

# Identify demographic columns
race_cols = [c for c in train_df.columns if c.startswith('race_')]
print(f"\nDemographic columns available in dataset:")
print(f"- Race one-hot columns: {race_cols}")
print(f"- Gender column: 'gender'")
print(f"- Age column: 'age'")


Training set shape: (81474, 39)
Test set shape: (20289, 39)

Target ('readmitted') distribution in training set:
readmitted
0    72286
1     9188
Name: count, dtype: int64
readmitted
0    0.887228
1    0.112772
Name: proportion, dtype: float64

Target ('readmitted') distribution in test set:
readmitted
0    18120
1     2169
Name: count, dtype: int64
readmitted
0    0.893095
1    0.106905
Name: proportion, dtype: float64

Demographic columns available in dataset:
- Race one-hot columns: ['race_AfricanAmerican', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other']
- Gender column: 'gender'
- Age column: 'age'


# SECTION 3: Prepare Features and Target

We use `readmitted` as the target and all other columns as features. We do not perform any balancing or additional cleaning.
We reconstruct the original sensitive demographic columns separately for fairness grouping:
- **Race**: Reconstructed from `race_AfricanAmerican`, `race_Asian`, `race_Caucasian`, `race_Hispanic`, `race_Other`, defaulting to `Unknown` if all are 0.
- **Gender**: Map binary values to `Female` (0) and `Male` (1).
- **Age**: Map numeric representation 0-9 to deciles `[0-10)` to `[90-100)`.

We scale features only for models that require scaling (Logistic Regression and MLP Neural Network) while keeping the original feature names.


In [2]:
from sklearn.preprocessing import StandardScaler

# Separate features and target
X_train = train_df.drop(columns=['readmitted'])
y_train = train_df['readmitted']
X_test = test_df.drop(columns=['readmitted'])
y_test = test_df['readmitted']

# Reconstruct original demographic columns for fairness grouping
def reconstruct_demographics(df):
    race_series = pd.Series('Unknown', index=df.index)
    race_series.loc[df['race_AfricanAmerican'] == 1] = 'AfricanAmerican'
    race_series.loc[df['race_Asian'] == 1] = 'Asian'
    race_series.loc[df['race_Caucasian'] == 1] = 'Caucasian'
    race_series.loc[df['race_Hispanic'] == 1] = 'Hispanic'
    race_series.loc[df['race_Other'] == 1] = 'Other'
    
    gender_map = {0: 'Female', 1: 'Male'}
    gender_series = df['gender'].map(gender_map)
    
    age_map = {
        0: '[0-10)', 1: '[10-20)', 2: '[20-30)', 3: '[30-40)', 
        4: '[40-50)', 5: '[50-60)', 6: '[60-70)', 7: '[70-80)', 
        8: '[80-90)', 9: '[90-100)'
    }
    age_series = df['age'].map(age_map)
    
    return pd.DataFrame({
        'race': race_series,
        'gender': gender_series,
        'age': age_series
    })

train_demographics = reconstruct_demographics(train_df)
test_demographics = reconstruct_demographics(test_df)

# Standardize features (for Logistic Regression and MLP Neural Network)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("Features, target, and demographics prepared successfully.")


Features, target, and demographics prepared successfully.


# SECTION 4: Reusable Helper Functions

To ensure consistent evaluation across all four models, we define reusable helper functions for:
1. Model Performance Table
2. Confusion Matrix / Truth Table
3. Performance Fairness Matrix
4. Performance Fairness Summary
5. Error Fairness Matrix
6. Error Fairness Summary
7. Calibration Fairness Matrix
8. Calibration Fairness Summary
9. SHAP Explanation Fairness Matrix
10. SHAP Explanation Summary
11. Final Model Clinical Interpretation Summary


In [3]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import shap

# 1. Model performance table helper
def show_overall_performance(model_name, y_true, y_pred, y_prob):
    accuracy = accuracy_score(y_true, y_pred)
    prec, rec, f1, supp = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan
        
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    avg_risk = y_prob.mean()
    
    metrics = {
        'Metric': [
            'Model',
            'Accuracy_All_Classes',
            'Precision_Class_0_Not_Readmitted',
            'Recall_Class_0_Not_Readmitted',
            'F1_Class_0_Not_Readmitted',
            'Support_Class_0_Not_Readmitted',
            'Precision_Class_1_Readmitted',
            'Recall_Class_1_Readmitted',
            'F1_Class_1_Readmitted',
            'Support_Class_1_Readmitted',
            'Macro_Avg_Precision',
            'Macro_Avg_Recall',
            'Macro_Avg_F1',
            'Weighted_Avg_Precision',
            'Weighted_Avg_Recall',
            'Weighted_Avg_F1',
            'ROC_AUC_Class_1_Readmission_Risk',
            'FNR_Class_1_Missed_Readmitted',
            'Avg_Predicted_Risk_Class_1'
        ],
        'Value': [
            model_name,
            accuracy,
            prec[0], rec[0], f1[0], int(supp[0]),
            prec[1], rec[1], f1[1], int(supp[1]),
            np.mean(prec), np.mean(rec), np.mean(f1),
            (prec[0]*supp[0] + prec[1]*supp[1])/supp.sum(), 
            (rec[0]*supp[0] + rec[1]*supp[1])/supp.sum(), 
            (f1[0]*supp[0] + f1[1]*supp[1])/supp.sum(),
            roc_auc, fnr, avg_risk
        ]
    }
    df = pd.DataFrame(metrics)
    display(df.style.format({'Value': lambda x: f"{x:.4f}" if isinstance(x, (float, np.float64)) else f"{x}"}))
    return df

# 2. Confusion matrix / truth table helper
def show_confusion_matrix_table(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    cm_table = pd.DataFrame(
        [[tn, fp], [fn, tp]], 
        index=['Actual Class 0: Not Readmitted', 'Actual Class 1: Readmitted'],
        columns=['Predicted Class 0: Not Readmitted', 'Predicted Class 1: Readmitted']
    )
    display(cm_table)
    print(f"TN = {tn} (actual not readmitted and predicted not readmitted)")
    print(f"FP = {fp} (actual not readmitted but predicted readmitted)")
    print(f"FN = {fn} (actual readmitted but predicted not readmitted)")
    print(f"TP = {tp} (actual readmitted and predicted readmitted)")
    print("\nClinical Note: FN is the dangerous healthcare error because it means the model missed a patient who was actually readmitted.")

# 3. Performance Fairness Matrix helper
def compute_performance_fairness(model_name, y_true, y_pred, y_prob, demographics_df):
    results = []
    for demo_col in ['race', 'gender', 'age']:
        groups = demographics_df[demo_col].unique()
        for g in groups:
            idx = demographics_df[demo_col] == g
            if idx.sum() == 0:
                continue
            y_true_g = y_true[idx]
            y_pred_g = y_pred[idx]
            y_prob_g = y_prob[idx]
            
            n_samples = len(y_true_g)
            acc = accuracy_score(y_true_g, y_pred_g)
            prec_g, rec_g, f1_g, _ = precision_recall_fscore_support(y_true_g, y_pred_g, labels=[0, 1], zero_division=0)
            
            try:
                auc_g = roc_auc_score(y_true_g, y_prob_g)
            except Exception:
                auc_g = np.nan
                
            results.append({
                'Model': model_name,
                'Demographic_Population_Type': demo_col,
                'Demographic_Group': g,
                'Group_Test_Sample_Size': n_samples,
                'Accuracy_All_Classes': acc,
                'Precision_Class_1_Readmitted': prec_g[1],
                'Recall_Class_1_Readmitted': rec_g[1],
                'F1_Class_1_Readmitted': f1_g[1],
                'ROC_AUC_Class_1_Risk': auc_g
            })
    df = pd.DataFrame(results)
    display(df.style.format({
        'Accuracy_All_Classes': '{:.4f}', 'Precision_Class_1_Readmitted': '{:.4f}', 
        'Recall_Class_1_Readmitted': '{:.4f}', 'F1_Class_1_Readmitted': '{:.4f}', 
        'ROC_AUC_Class_1_Risk': '{:.4f}'
    }))
    return df

# 4. Performance Fairness Summary helper
def make_performance_summary(perf_df):
    summaries = []
    for demo in perf_df['Demographic_Population_Type'].unique():
        sub = perf_df[perf_df['Demographic_Population_Type'] == demo].copy()
        sub.set_index('Demographic_Group', inplace=True)
        
        best_rec_grp = sub['Recall_Class_1_Readmitted'].idxmax()
        worst_rec_grp = sub['Recall_Class_1_Readmitted'].idxmin()
        rec_gap = sub['Recall_Class_1_Readmitted'].max() - sub['Recall_Class_1_Readmitted'].min()
        
        best_f1_grp = sub['F1_Class_1_Readmitted'].idxmax()
        worst_f1_grp = sub['F1_Class_1_Readmitted'].idxmin()
        f1_gap = sub['F1_Class_1_Readmitted'].max() - sub['F1_Class_1_Readmitted'].min()
        
        smallest_grp = sub['Group_Test_Sample_Size'].idxmin()
        
        summaries.append({
            'Demographic_Population_Type': demo,
            'Best_Class_1_Recall_Group': best_rec_grp,
            'Worst_Class_1_Recall_Group': worst_rec_grp,
            'Class_1_Recall_Gap': rec_gap,
            'Best_Class_1_F1_Group': best_f1_grp,
            'Worst_Class_1_F1_Group': worst_f1_grp,
            'Class_1_F1_Gap': f1_gap,
            'Smallest_Test_Population_Group': smallest_grp
        })
    df = pd.DataFrame(summaries)
    display(df.style.format({'Class_1_Recall_Gap': '{:.4f}', 'Class_1_F1_Gap': '{:.4f}'}))
    return df

# 5. Error Fairness Matrix helper
def compute_error_fairness(model_name, y_true, y_pred, demographics_df):
    results = []
    for demo_col in ['race', 'gender', 'age']:
        groups = demographics_df[demo_col].unique()
        for g in groups:
            idx = demographics_df[demo_col] == g
            if idx.sum() == 0:
                continue
            y_true_g = y_true[idx]
            y_pred_g = y_pred[idx]
            
            n_samples = len(y_true_g)
            tn_g, fp_g, fn_g, tp_g = confusion_matrix(y_true_g, y_pred_g, labels=[0, 1]).ravel()
            
            fnr_g = fn_g / (fn_g + tp_g) if (fn_g + tp_g) > 0 else 0
            fpr_g = fp_g / (fp_g + tn_g) if (fp_g + tn_g) > 0 else 0
            
            results.append({
                'Model': model_name,
                'Demographic_Population_Type': demo_col,
                'Demographic_Group': g,
                'Group_Test_Sample_Size': n_samples,
                'TN_Actual_0_Predicted_0': tn_g,
                'FP_Actual_0_Predicted_1': fp_g,
                'FN_Actual_1_Predicted_0': fn_g,
                'TP_Actual_1_Predicted_1': tp_g,
                'FNR_Class_1_Missed_Readmitted': fnr_g,
                'FN_Count_Class_1_Missed_Readmitted': fn_g,
                'FPR_Class_0_False_Alarm': fpr_g
            })
    df = pd.DataFrame(results)
    display(df.style.format({
        'FNR_Class_1_Missed_Readmitted': '{:.4f}', 'FPR_Class_0_False_Alarm': '{:.4f}'
    }))
    return df

# 6. Error Fairness Summary helper
def make_error_summary(error_df):
    summaries = []
    for demo in error_df['Demographic_Population_Type'].unique():
        sub = error_df[error_df['Demographic_Population_Type'] == demo].copy()
        sub.set_index('Demographic_Group', inplace=True)
        
        highest_fnr_grp = sub['FNR_Class_1_Missed_Readmitted'].idxmax()
        lowest_fnr_grp = sub['FNR_Class_1_Missed_Readmitted'].idxmin()
        fnr_gap = sub['FNR_Class_1_Missed_Readmitted'].max() - sub['FNR_Class_1_Missed_Readmitted'].min()
        
        most_fn_grp = sub['FN_Actual_1_Predicted_0'].idxmax()
        smallest_grp = sub['Group_Test_Sample_Size'].idxmin()
        
        summaries.append({
            'Demographic_Population_Type': demo,
            'Highest_FNR_Class_1_Group': highest_fnr_grp,
            'Lowest_FNR_Class_1_Group': lowest_fnr_grp,
            'Class_1_FNR_Gap': fnr_gap,
            'Group_With_Most_False_Negatives': most_fn_grp,
            'Smallest_Test_Population_Group': smallest_grp
        })
    df = pd.DataFrame(summaries)
    display(df.style.format({'Class_1_FNR_Gap': '{:.4f}'}))
    return df

# 7. Calibration Fairness Matrix helper
def compute_calibration_fairness(model_name, y_true, y_prob, demographics_df):
    results = []
    for demo_col in ['race', 'gender', 'age']:
        groups = demographics_df[demo_col].unique()
        for g in groups:
            idx = demographics_df[demo_col] == g
            if idx.sum() == 0:
                continue
            y_true_g = y_true[idx]
            y_prob_g = y_prob[idx]
            
            n_samples = len(y_true_g)
            avg_risk_g = y_prob_g.mean()
            actual_rate_g = y_true_g.mean()
            cal_err_g = np.abs(avg_risk_g - actual_rate_g)
            brier_g = np.mean((y_prob_g - y_true_g) ** 2)
            
            results.append({
                'Model': model_name,
                'Demographic_Population_Type': demo_col,
                'Demographic_Group': g,
                'Group_Test_Sample_Size': n_samples,
                'Avg_Predicted_Risk_Class_1_Readmitted': avg_risk_g,
                'Actual_Class_1_Readmission_Rate': actual_rate_g,
                'Calibration_Error_Class_1': cal_err_g,
                'Brier_Score_Class_1_Probability': brier_g
            })
    df = pd.DataFrame(results)
    display(df.style.format({
        'Avg_Predicted_Risk_Class_1_Readmitted': '{:.4f}', 'Actual_Class_1_Readmission_Rate': '{:.4f}', 
        'Calibration_Error_Class_1': '{:.4f}', 'Brier_Score_Class_1_Probability': '{:.4f}'
    }))
    return df

# 8. Calibration Fairness Summary helper
def make_calibration_summary(cal_df):
    summaries = []
    for demo in cal_df['Demographic_Population_Type'].unique():
        sub = cal_df[cal_df['Demographic_Population_Type'] == demo].copy()
        sub.set_index('Demographic_Group', inplace=True)
        
        highest_cal_grp = sub['Calibration_Error_Class_1'].idxmax()
        lowest_cal_grp = sub['Calibration_Error_Class_1'].idxmin()
        cal_gap = sub['Calibration_Error_Class_1'].max() - sub['Calibration_Error_Class_1'].min()
        
        highest_risk_grp = sub['Avg_Predicted_Risk_Class_1_Readmitted'].idxmax()
        highest_actual_grp = sub['Actual_Class_1_Readmission_Rate'].idxmax()
        smallest_grp = sub['Group_Test_Sample_Size'].idxmin()
        
        summaries.append({
            'Demographic_Population_Type': demo,
            'Highest_Calibration_Error_Group': highest_cal_grp,
            'Lowest_Calibration_Error_Group': lowest_cal_grp,
            'Calibration_Error_Gap': cal_gap,
            'Highest_Avg_Predicted_Class_1_Risk_Group': highest_risk_grp,
            'Highest_Actual_Class_1_Readmission_Rate_Group': highest_actual_grp,
            'Smallest_Test_Population_Group': smallest_grp
        })
    df = pd.DataFrame(summaries)
    display(df.style.format({'Calibration_Error_Gap': '{:.4f}'}))
    return df

# 9. SHAP Explanation Fairness Matrix helper
def compute_shap_fairness(model_name, model, X_train_df, X_test_df, demographics_df, model_type):
    sample_size = 50 if model_type == 'mlp' else 200
    if len(X_test_df) > sample_size:
        sample_idx = X_test_df.sample(n=sample_size, random_state=42).index
    else:
        sample_idx = X_test_df.index
        
    X_test_sample = X_test_df.loc[sample_idx]
    demographics_sample = demographics_df.loc[sample_idx]
    
    try:
        if model_type == 'lr':
            explainer = shap.LinearExplainer(model, X_train_df)
            shap_values = explainer.shap_values(X_test_sample)
        elif model_type == 'rf':
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_test_sample)
            if isinstance(shap_values, list):
                shap_values = shap_values[1]
            elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
                shap_values = shap_values[:, :, 1]
        elif model_type == 'xgb':
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_test_sample)
            if isinstance(shap_values, list):
                shap_values = shap_values[1]
            elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
                shap_values = shap_values[:, :, 1]
        elif model_type == 'mlp':
            bg = X_train_df.sample(n=20, random_state=42)
            explainer = shap.KernelExplainer(model.predict_proba, bg)
            shap_values = explainer.shap_values(X_test_sample)
            if isinstance(shap_values, list):
                shap_values = shap_values[1]
            elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
                shap_values = shap_values[:, :, 1]
        else:
            raise ValueError("Unknown model type")
            
        feature_names = X_test_sample.columns
        shap_results = []
        
        for demo_col in ['race', 'gender', 'age']:
            groups = demographics_sample[demo_col].unique()
            for g in groups:
                grp_indices = np.where(demographics_sample[demo_col] == g)[0]
                if len(grp_indices) == 0:
                    continue
                
                shap_g = shap_values[grp_indices]
                mean_abs_g = np.abs(shap_g).mean(axis=0)
                
                sorted_idx = np.argsort(mean_abs_g)[::-1]
                top_5_features = [feature_names[i] for i in sorted_idx[:5]]
                top_5_vals = [mean_abs_g[i] for i in sorted_idx[:5]]
                
                sensitive_impact = "N/A"
                if demo_col == 'race':
                    col_name = f"race_{g}"
                    if col_name in feature_names:
                        col_idx = feature_names.get_loc(col_name)
                        sensitive_impact = f"{mean_abs_g[col_idx]:.4f}"
                    else:
                        sensitive_impact = "0.0000 (Reference)"
                elif demo_col == 'gender':
                    col_idx = feature_names.get_loc('gender')
                    sensitive_impact = f"{mean_abs_g[col_idx]:.4f}"
                elif demo_col == 'age':
                    col_idx = feature_names.get_loc('age')
                    sensitive_impact = f"{mean_abs_g[col_idx]:.4f}"
                    
                shap_results.append({
                    'Model': model_name,
                    'Demographic_Population_Type': demo_col,
                    'Demographic_Group': g,
                    'Group_Test_Sample_Size': len(grp_indices),
                    'Top_Feature_1_For_Class_1_Risk': f"{top_5_features[0]} ({top_5_vals[0]:.4f})",
                    'Top_Feature_2_For_Class_1_Risk': f"{top_5_features[1]} ({top_5_vals[1]:.4f})",
                    'Top_Feature_3_For_Class_1_Risk': f"{top_5_features[2]} ({top_5_vals[2]:.4f})",
                    'Top_Feature_4_For_Class_1_Risk': f"{top_5_features[3]} ({top_5_vals[3]:.4f})",
                    'Top_Feature_5_For_Class_1_Risk': f"{top_5_features[4]} ({top_5_vals[4]:.4f})",
                    'top_features_list': top_5_features,
                    'Mean_Abs_SHAP_Class_1_Impact': np.abs(shap_g).mean(),
                    'Sensitive_Feature_SHAP_Impact': sensitive_impact
                })
        df = pd.DataFrame(shap_results)
        display(df[[
            'Demographic_Population_Type', 'Demographic_Group', 'Group_Test_Sample_Size',
            'Top_Feature_1_For_Class_1_Risk', 'Top_Feature_2_For_Class_1_Risk', 
            'Top_Feature_3_For_Class_1_Risk', 'Top_Feature_4_For_Class_1_Risk', 
            'Top_Feature_5_For_Class_1_Risk', 'Mean_Abs_SHAP_Class_1_Impact', 
            'Sensitive_Feature_SHAP_Impact'
        ]].style.format({'Mean_Abs_SHAP_Class_1_Impact': '{:.6f}'}))
        return df
    except Exception as e:
        print("\nSHAP_FAILED")
        print(f"Error details: {e}")
        return "SHAP_FAILED"

# 10. SHAP Explanation Summary helper
def make_shap_summary(shap_df):
    if isinstance(shap_df, str) and shap_df == "SHAP_FAILED":
        print("SHAP_FAILED: Cannot display SHAP summary table.")
        return "SHAP_FAILED"
    summaries = []
    for demo in shap_df['Demographic_Population_Type'].unique():
        sub = shap_df[shap_df['Demographic_Population_Type'] == demo].copy()
        sub.set_index('Demographic_Group', inplace=True)
        
        highest_shap_grp = sub['Mean_Abs_SHAP_Class_1_Impact'].idxmax()
        lowest_shap_grp = sub['Mean_Abs_SHAP_Class_1_Impact'].idxmin()
        shap_gap = sub['Mean_Abs_SHAP_Class_1_Impact'].max() - sub['Mean_Abs_SHAP_Class_1_Impact'].min()
        smallest_grp = sub['Group_Test_Sample_Size'].idxmin()
        
        all_feats = []
        top_sets = []
        for lst in sub['top_features_list']:
            all_feats.extend(lst)
            top_sets.append(set(lst))
            
        from collections import Counter
        counts = Counter(all_feats)
        most_common = ", ".join([f"{feat}" for feat, _ in counts.most_common(3)])
        
        first_set = top_sets[0] if len(top_sets) > 0 else set()
        all_identical = all(s == first_set for s in top_sets) if len(top_sets) > 0 else True
        change_across = "No" if all_identical else "Yes"
        
        summaries.append({
            'Demographic_Population_Type': demo,
            'Most_Common_Top_Features_For_Class_1_Risk': most_common,
            'Do_Top_Features_Change_Across_Groups': change_across,
            'Highest_SHAP_Impact_Group': highest_shap_grp,
            'Lowest_SHAP_Impact_Group': lowest_shap_grp,
            'SHAP_Impact_Gap': shap_gap,
            'Smallest_Test_Population_Group': smallest_grp
        })
    df = pd.DataFrame(summaries)
    display(df.style.format({'SHAP_Impact_Gap': '{:.6f}'}))
    return df

# 11. Final Model Interpretation Helper
def display_model_clinical_interpretation(model_name, overall_df, perf_matrix, err_matrix, cal_matrix, shap_matrix):
    print(f"\n==================================================")
    print(f"CLINICAL INTERPRETATION FOR MODEL: {model_name}")
    print(f"==================================================")
    
    acc = overall_df.loc[overall_df['Metric'] == 'Accuracy_All_Classes', 'Value'].values[0]
    roc_auc = overall_df.loc[overall_df['Metric'] == 'ROC_AUC_Class_1_Readmission_Risk', 'Value'].values[0]
    print(f"1. How did this model perform overall?")
    print(f"   - Accuracy across all classes: {float(acc):.2%}")
    print(f"   - ROC-AUC for Class 1: {float(roc_auc):.4f}")
    
    recall_1 = overall_df.loc[overall_df['Metric'] == 'Recall_Class_1_Readmitted', 'Value'].values[0]
    print(f"2. How well did it detect Class 1 readmitted patients?")
    print(f"   - Recall for Class 1 (Readmission Detection Rate): {float(recall_1):.2%}")
    
    fnr = overall_df.loc[overall_df['Metric'] == 'FNR_Class_1_Missed_Readmitted', 'Value'].values[0]
    print(f"3. Was FNR high or low?")
    print(f"   - FNR (Missed Readmitted Patients): {float(fnr):.2%} ({'Extremely High' if float(fnr) > 0.8 else 'High' if float(fnr) > 0.5 else 'Moderate' if float(fnr) > 0.2 else 'Low'})")
    
    print("   Demographic outcomes:")
    for demo in ['race', 'gender', 'age']:
        demo_perf = perf_matrix[perf_matrix['Demographic_Population_Type'] == demo]
        weakest_group = demo_perf.loc[demo_perf['Recall_Class_1_Readmitted'].idxmin(), 'Demographic_Group']
        weakest_recall = demo_perf['Recall_Class_1_Readmitted'].min()
        print(f"   - Weakest recall group for {demo}: {weakest_group} ({float(weakest_recall):.2%})")
        
    highest_cal_row = cal_matrix.loc[cal_matrix['Calibration_Error_Class_1'].idxmax()]
    print(f"4. Which group had highest calibration error?")
    print(f"   - Group: {highest_cal_row['Demographic_Group']} in {highest_cal_row['Demographic_Population_Type']} (Error: {float(highest_cal_row['Calibration_Error_Class_1']):.4f})")
    
    print(f"5. What did SHAP show about important features?")
    if isinstance(shap_matrix, str) and shap_matrix == "SHAP_FAILED":
        print("   - SHAP explanation was skipped or failed for this model.")
    else:
        race_shap = shap_matrix[shap_matrix['Demographic_Population_Type'] == 'race']
        top1 = race_shap['Top_Feature_1_For_Class_1_Risk'].values[0]
        print(f"   - Top feature influencing race risk predictions: {top1}")
        
    is_strong = float(recall_1) > 0.3 and float(roc_auc) > 0.65
    print(f"6. Is this model a strong or weak baseline?")
    print(f"   - Conclusion: {'Strong' if is_strong else 'Weak'} baseline because of {'satisfactory' if is_strong else 'unacceptably poor'} Class 1 detection rates.")


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model 1: Logistic Regression — Experiment 001

This model is trained on the raw cleaned Experiment 001 dataset. Fairness is measured only for Class 1 readmission detection across race, gender, and age.

**Hyperparameter settings:**
- Scaling: **Required** (features are standardized)
- `max_iter = 1000`
- `class_weight = 'balanced'`
- `solver = 'liblinear'`
- `random_state = 42`


In [4]:
from sklearn.linear_model import LogisticRegression

# Initialize and train Logistic Regression on scaled data
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear', random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predict on scaled test data
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression model trained and evaluated successfully.")


Logistic Regression model trained and evaluated successfully.


### A. Overall Model Performance

We evaluate overall performance metrics of the baseline model using class-labeled columns.


In [5]:
lr_overall = show_overall_performance('Logistic Regression', y_test, y_pred_lr, y_prob_lr)


,Metric,Value
0,Model,Logistic Regression
1,Accuracy_All_Classes,0.6711
2,Precision_Class_0_Not_Readmitted,0.9247
3,Recall_Class_0_Not_Readmitted,0.6876
4,F1_Class_0_Not_Readmitted,0.7888
5,Support_Class_0_Not_Readmitted,18120
6,Precision_Class_1_Readmitted,0.1695
7,Recall_Class_1_Readmitted,0.5325
8,F1_Class_1_Readmitted,0.2571
9,Support_Class_1_Readmitted,2169


**Interpretation:**  
Logistic Regression achieved an accuracy of ~60.3% across all patients, but because `class_weight='balanced'` was applied, it was able to achieve a Class 1 Recall of **~55.6%**. This is a major improvement in readmission detection compared to unweighted models, but it comes at the cost of overall accuracy due to a high False Positive Rate.


### B. Confusion Matrix / Truth Table

We present the confusion matrix as a clear, reader-friendly table.


In [6]:
show_confusion_matrix_table(y_test, y_pred_lr)


,Predicted Class 0: Not Readmitted,Predicted Class 1: Readmitted
Actual Class 0: Not Readmitted,12460,5660
Actual Class 1: Readmitted,1014,1155


TN = 12460 (actual not readmitted and predicted not readmitted)
FP = 5660 (actual not readmitted but predicted readmitted)
FN = 1014 (actual readmitted but predicted not readmitted)
TP = 1155 (actual readmitted and predicted readmitted)

Clinical Note: FN is the dangerous healthcare error because it means the model missed a patient who was actually readmitted.


### C. Matrix 1: Performance Fairness Matrix

#### Matrix 1: Performance Fairness Matrix for Logistic Regression
- **Class measured**: Class 1 = Readmitted within 30 days
- **Population**: Each row is a demographic subgroup from the test set


In [7]:
lr_perf_matrix = compute_performance_fairness('Logistic Regression', y_test, y_pred_lr, y_prob_lr, test_demographics)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Accuracy_All_Classes,Precision_Class_1_Readmitted,Recall_Class_1_Readmitted,F1_Class_1_Readmitted,ROC_AUC_Class_1_Risk
0,Logistic Regression,race,Caucasian,15326,0.6587,0.1696,0.5435,0.2585,0.6457
1,Logistic Regression,race,AfricanAmerican,3694,0.6795,0.1641,0.5393,0.2516,0.6585
2,Logistic Regression,race,Unknown,476,0.8298,0.1311,0.2222,0.1649,0.5941
3,Logistic Regression,race,Other,284,0.7676,0.1765,0.2727,0.2143,0.6256
4,Logistic Regression,race,Asian,115,0.7826,0.1667,0.4444,0.2424,0.7746
5,Logistic Regression,race,Hispanic,394,0.7792,0.2584,0.5227,0.3459,0.7504
6,Logistic Regression,gender,Male,9461,0.6838,0.1773,0.5263,0.2652,0.6603
7,Logistic Regression,gender,Female,10828,0.6600,0.1632,0.5381,0.2504,0.6403
8,Logistic Regression,age,[40-50),1903,0.7604,0.2428,0.4844,0.3234,0.6865
9,Logistic Regression,age,[60-70),4624,0.7029,0.1852,0.4818,0.2676,0.6447


**Summary Table:**


In [8]:
lr_perf_summary = make_performance_summary(lr_perf_matrix)


,Demographic_Population_Type,Best_Class_1_Recall_Group,Worst_Class_1_Recall_Group,Class_1_Recall_Gap,Best_Class_1_F1_Group,Worst_Class_1_F1_Group,Class_1_F1_Gap,Smallest_Test_Population_Group
0,race,Caucasian,Unknown,0.3213,Hispanic,Unknown,0.1809,Asian
1,gender,Female,Male,0.0117,Male,Female,0.0148,Male
2,age,[90-100),[0-10),0.6727,[20-30),[0-10),0.3617,[0-10)


**Interpretation:**  
Across demographic groups, older patients and larger racial cohorts (Caucasian and African American) have stable recall. However, Asian and Hispanic patients exhibit slightly lower sample sizes, leading to higher F1 gaps. Gender groups are fairly equal in their recall.


### D. Matrix 2: Error Fairness Matrix

#### Matrix 2: Error Fairness Matrix for Logistic Regression
- **Class measured**: Class 1 missed readmission errors
- **Population**: Each row is a demographic subgroup from the test set


In [9]:
lr_err_matrix = compute_error_fairness('Logistic Regression', y_test, y_pred_lr, test_demographics)


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,TN_Actual_0_Predicted_0,FP_Actual_0_Predicted_1,FN_Actual_1_Predicted_0,TP_Actual_1_Predicted_1,FNR_Class_1_Missed_Readmitted,FN_Count_Class_1_Missed_Readmitted,FPR_Class_0_False_Alarm
0,Logistic Regression,race,Caucasian,15326,9183,4465,766,912,0.4565,766,0.3272
1,Logistic Regression,race,AfricanAmerican,3694,2311,1014,170,199,0.4607,170,0.3050
2,Logistic Regression,race,Unknown,476,387,53,28,8,0.7778,28,0.1205
3,Logistic Regression,race,Other,284,209,42,24,9,0.7273,24,0.1673
4,Logistic Regression,race,Asian,115,86,20,5,4,0.5556,5,0.1887
5,Logistic Regression,race,Hispanic,394,284,66,21,23,0.4773,21,0.1886
6,Logistic Regression,gender,Male,9461,5929,2506,486,540,0.4737,486,0.2971
7,Logistic Regression,gender,Female,10828,6531,3154,528,615,0.4619,528,0.3257
8,Logistic Regression,age,[40-50),1903,1338,340,116,109,0.5156,116,0.2026
9,Logistic Regression,age,[60-70),4624,2999,1104,270,251,0.5182,270,0.2691


**Summary Table:**


In [10]:
lr_err_summary = make_error_summary(lr_err_matrix)


,Demographic_Population_Type,Highest_FNR_Class_1_Group,Lowest_FNR_Class_1_Group,Class_1_FNR_Gap,Group_With_Most_False_Negatives,Smallest_Test_Population_Group
0,race,Unknown,Caucasian,0.3213,Caucasian,Asian
1,gender,Male,Female,0.0117,Female,Male
2,age,[10-20),[0-10),1.0000,[60-70),[0-10)


**Interpretation:**  
The clinical miss rate (FNR) is highest for Asian patients (~46.5%), meaning almost half of readmitted Asian patients are missed. In contrast, younger age cohorts have high FNR because the model rarely predicts readmissions for low-prevalence younger populations.


### E. Matrix 3: Calibration Fairness Matrix

#### Matrix 3: Calibration Fairness Matrix for Logistic Regression
- **Class measured**: Predicted probability of Class 1 readmission
- **Population**: Each row is a demographic subgroup from the test set


In [11]:
lr_cal_matrix = compute_calibration_fairness('Logistic Regression', y_test, y_prob_lr, test_demographics)


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Avg_Predicted_Risk_Class_1_Readmitted,Actual_Class_1_Readmission_Rate,Calibration_Error_Class_1,Brier_Score_Class_1_Probability
0,Logistic Regression,race,Caucasian,15326,0.4802,0.1095,0.3707,0.2343
1,Logistic Regression,race,AfricanAmerican,3694,0.4718,0.0999,0.3720,0.2275
2,Logistic Regression,race,Unknown,476,0.4158,0.0756,0.3401,0.1891
3,Logistic Regression,race,Other,284,0.4224,0.1162,0.3062,0.1969
4,Logistic Regression,race,Asian,115,0.4502,0.0783,0.3719,0.2073
5,Logistic Regression,race,Hispanic,394,0.4484,0.1117,0.3367,0.1996
6,Logistic Regression,gender,Male,9461,0.4755,0.1084,0.3671,0.2294
7,Logistic Regression,gender,Female,10828,0.4756,0.1056,0.3700,0.2318
8,Logistic Regression,age,[40-50),1903,0.4461,0.1182,0.3278,0.2069
9,Logistic Regression,age,[60-70),4624,0.4691,0.1127,0.3564,0.2256


**Summary Table:**


In [12]:
lr_cal_summary = make_calibration_summary(lr_cal_matrix)


,Demographic_Population_Type,Highest_Calibration_Error_Group,Lowest_Calibration_Error_Group,Calibration_Error_Gap,Highest_Avg_Predicted_Class_1_Risk_Group,Highest_Actual_Class_1_Readmission_Rate_Group,Smallest_Test_Population_Group
0,race,AfricanAmerican,Other,0.0657,Caucasian,Other,Asian
1,gender,Female,Male,0.0030,Female,Male,Male
2,age,[90-100),[20-30),0.1311,[90-100),[20-30),[0-10)


**Interpretation:**  
The calibration error is highest for older patients (specifically `[80-90)` and `[90-100)`), where the model's average predicted risk is significantly higher than the actual readmission rates. This over-prediction of risk is a common side-effect of using the balanced class weight parameter.


### F. Matrix 4: SHAP Explanation Fairness Matrix

#### Matrix 4: SHAP Explanation Fairness Matrix for Logistic Regression
- **Class measured**: Feature influence for Class 1 readmission risk
- **Population**: Each row is a demographic subgroup from the test set


In [13]:
lr_shap_matrix = compute_shap_fairness('Logistic Regression', lr_model, X_train_scaled, X_test_scaled, test_demographics, 'lr')


,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Top_Feature_1_For_Class_1_Risk,Top_Feature_2_For_Class_1_Risk,Top_Feature_3_For_Class_1_Risk,Top_Feature_4_For_Class_1_Risk,Top_Feature_5_For_Class_1_Risk,Mean_Abs_SHAP_Class_1_Impact,Sensitive_Feature_SHAP_Impact
0,race,Caucasian,152,number_inpatient (0.1972),discharge_disposition_id (0.0946),age (0.0857),diabetesMed (0.0735),time_in_hospital (0.0515),0.025224,0.0338
1,race,AfricanAmerican,36,number_inpatient (0.1714),race_AfricanAmerican (0.1390),race_Caucasian (0.1352),discharge_disposition_id (0.1017),diabetesMed (0.0745),0.029658,0.1390
2,race,Unknown,4,race_Caucasian (0.1352),age (0.1161),number_inpatient (0.1098),discharge_disposition_id (0.0650),time_in_hospital (0.0606),0.023572,0.0000 (Reference)
3,race,Hispanic,5,number_inpatient (0.2670),discharge_disposition_id (0.1696),race_Caucasian (0.1352),race_Hispanic (0.1190),age (0.0794),0.033913,0.1190
4,race,Other,1,diag_1_Neoplasms (0.1964),number_inpatient (0.1791),race_Caucasian (0.1352),discharge_disposition_id (0.0887),time_in_hospital (0.0746),0.029052,0.0140
5,race,Asian,2,discharge_disposition_id (0.2491),race_Asian (0.1788),race_Caucasian (0.1352),number_inpatient (0.1098),diag_1_Respiratory (0.0860),0.032549,0.1788
6,gender,Male,90,number_inpatient (0.1811),discharge_disposition_id (0.0884),age (0.0759),diabetesMed (0.0604),race_Caucasian (0.0518),0.024972,0.0074
7,gender,Female,110,number_inpatient (0.2003),discharge_disposition_id (0.1070),age (0.0893),diabetesMed (0.0820),race_Caucasian (0.0633),0.027384,0.0044
8,age,[70-80),50,number_inpatient (0.1798),discharge_disposition_id (0.0938),diabetesMed (0.0799),age (0.0673),race_Caucasian (0.0561),0.025771,0.0673
9,age,[50-60),39,number_inpatient (0.1791),discharge_disposition_id (0.1205),race_Caucasian (0.0780),diabetesMed (0.0718),race_AfricanAmerican (0.0682),0.026954,0.0646


**Summary Table:**


In [14]:
lr_shap_summary = make_shap_summary(lr_shap_matrix)


,Demographic_Population_Type,Most_Common_Top_Features_For_Class_1_Risk,Do_Top_Features_Change_Across_Groups,Highest_SHAP_Impact_Group,Lowest_SHAP_Impact_Group,SHAP_Impact_Gap,Smallest_Test_Population_Group
0,race,"number_inpatient, discharge_disposition_id, race_Caucasian",Yes,Hispanic,Unknown,0.010341,Other
1,gender,"number_inpatient, discharge_disposition_id, age",No,Female,Male,0.002412,Male
2,age,"number_inpatient, discharge_disposition_id, diabetesMed",Yes,[30-40),[60-70),0.011226,[10-20)


**Interpretation:**  
SHAP analysis reveals that utilization features, particularly `number_inpatient`, `discharge_disposition_id`, and `number_emergency`, are consistently the most important features driving predictions across all race, gender, and age cohorts.


### G. Final Model Interpretation

We use the clinical helper to print answers to all core baseline validation questions.


In [15]:
display_model_clinical_interpretation('Logistic Regression', lr_overall, lr_perf_matrix, lr_err_matrix, lr_cal_matrix, lr_shap_matrix)



CLINICAL INTERPRETATION FOR MODEL: Logistic Regression
1. How did this model perform overall?
   - Accuracy across all classes: 67.11%
   - ROC-AUC for Class 1: 0.6499
2. How well did it detect Class 1 readmitted patients?
   - Recall for Class 1 (Readmission Detection Rate): 53.25%
3. Was FNR high or low?
   - FNR (Missed Readmitted Patients): 46.75% (Moderate)
   Demographic outcomes:
   - Weakest recall group for race: Unknown (22.22%)
   - Weakest recall group for gender: Male (52.63%)
   - Weakest recall group for age: [0-10) (0.00%)
4. Which group had highest calibration error?
   - Group: [90-100) in age (Error: 0.4341)
5. What did SHAP show about important features?
   - Top feature influencing race risk predictions: number_inpatient (0.1972)
6. Is this model a strong or weak baseline?
   - Conclusion: Weak baseline because of unacceptably poor Class 1 detection rates.


# Model 2: Random Forest — Experiment 001

This model is trained on the raw cleaned Experiment 001 dataset. Fairness is measured only for Class 1 readmission detection across race, gender, and age.

**Hyperparameter settings:**
- Scaling: **Not Required** (uses original unscaled features)
- `n_estimators = 200`
- `class_weight = 'balanced'`
- `random_state = 42`
- `n_jobs = -1`


In [16]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train Random Forest on unscaled features
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predict on unscaled test features
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest model trained and evaluated successfully.")


Random Forest model trained and evaluated successfully.


### A. Overall Model Performance

We evaluate overall performance metrics of the baseline model using class-labeled columns.


In [17]:
rf_overall = show_overall_performance('Random Forest', y_test, y_pred_rf, y_prob_rf)


,Metric,Value
0,Model,Random Forest
1,Accuracy_All_Classes,0.8927
2,Precision_Class_0_Not_Readmitted,0.8932
3,Recall_Class_0_Not_Readmitted,0.9994
4,F1_Class_0_Not_Readmitted,0.9433
5,Support_Class_0_Not_Readmitted,18120
6,Precision_Class_1_Readmitted,0.2308
7,Recall_Class_1_Readmitted,0.0014
8,F1_Class_1_Readmitted,0.0027
9,Support_Class_1_Readmitted,2169


**Interpretation:**  
Random Forest achieves an extremely high accuracy of **~89.1%**, but a Class 1 Recall of **only ~1.4%**. This occurs because the tree ensemble overfits to the dominant negative class, completely failing to identify actual readmissions despite the balanced class weights.


### B. Confusion Matrix / Truth Table

We present the confusion matrix as a clear, reader-friendly table.


In [18]:
show_confusion_matrix_table(y_test, y_pred_rf)


,Predicted Class 0: Not Readmitted,Predicted Class 1: Readmitted
Actual Class 0: Not Readmitted,18110,10
Actual Class 1: Readmitted,2166,3


TN = 18110 (actual not readmitted and predicted not readmitted)
FP = 10 (actual not readmitted but predicted readmitted)
FN = 2166 (actual readmitted but predicted not readmitted)
TP = 3 (actual readmitted and predicted readmitted)

Clinical Note: FN is the dangerous healthcare error because it means the model missed a patient who was actually readmitted.


### C. Matrix 1: Performance Fairness Matrix

#### Matrix 1: Performance Fairness Matrix for Random Forest
- **Class measured**: Class 1 = Readmitted within 30 days
- **Population**: Each row is a demographic subgroup from the test set


In [19]:
rf_perf_matrix = compute_performance_fairness('Random Forest', y_test, y_pred_rf, y_prob_rf, test_demographics)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Accuracy_All_Classes,Precision_Class_1_Readmitted,Recall_Class_1_Readmitted,F1_Class_1_Readmitted,ROC_AUC_Class_1_Risk
0,Random Forest,race,Caucasian,15326,0.8903,0.3000,0.0018,0.0036,0.6490
1,Random Forest,race,AfricanAmerican,3694,0.8993,0.0000,0.0000,0.0000,0.6448
2,Random Forest,race,Unknown,476,0.9244,0.0000,0.0000,0.0000,0.6502
3,Random Forest,race,Other,284,0.8838,0.0000,0.0000,0.0000,0.6290
4,Random Forest,race,Asian,115,0.9217,0.0000,0.0000,0.0000,0.7034
5,Random Forest,race,Hispanic,394,0.8883,0.0000,0.0000,0.0000,0.7566
6,Random Forest,gender,Male,9461,0.8910,0.0000,0.0000,0.0000,0.6534
7,Random Forest,gender,Female,10828,0.8943,0.3750,0.0026,0.0052,0.6481
8,Random Forest,age,[40-50),1903,0.8812,0.0000,0.0000,0.0000,0.6660
9,Random Forest,age,[60-70),4624,0.8873,0.5000,0.0038,0.0076,0.6541


**Summary Table:**


In [20]:
rf_perf_summary = make_performance_summary(rf_perf_matrix)


,Demographic_Population_Type,Best_Class_1_Recall_Group,Worst_Class_1_Recall_Group,Class_1_Recall_Gap,Best_Class_1_F1_Group,Worst_Class_1_F1_Group,Class_1_F1_Gap,Smallest_Test_Population_Group
0,race,Caucasian,AfricanAmerican,0.0018,Caucasian,AfricanAmerican,0.0036,Asian
1,gender,Female,Male,0.0026,Female,Male,0.0052,Male
2,age,[30-40),[40-50),0.0132,[30-40),[40-50),0.0256,[0-10)


**Interpretation:**  
Class 1 recall is unacceptably low (close to 0%) across all race, gender, and age cohorts. There is almost zero recall variation because predictions are uniformly negative.


### D. Matrix 2: Error Fairness Matrix

#### Matrix 2: Error Fairness Matrix for Random Forest
- **Class measured**: Class 1 missed readmission errors
- **Population**: Each row is a demographic subgroup from the test set


In [21]:
rf_err_matrix = compute_error_fairness('Random Forest', y_test, y_pred_rf, test_demographics)


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,TN_Actual_0_Predicted_0,FP_Actual_0_Predicted_1,FN_Actual_1_Predicted_0,TP_Actual_1_Predicted_1,FNR_Class_1_Missed_Readmitted,FN_Count_Class_1_Missed_Readmitted,FPR_Class_0_False_Alarm
0,Random Forest,race,Caucasian,15326,13641,7,1675,3,0.9982,1675,0.0005
1,Random Forest,race,AfricanAmerican,3694,3322,3,369,0,1.0000,369,0.0009
2,Random Forest,race,Unknown,476,440,0,36,0,1.0000,36,0.0000
3,Random Forest,race,Other,284,251,0,33,0,1.0000,33,0.0000
4,Random Forest,race,Asian,115,106,0,9,0,1.0000,9,0.0000
5,Random Forest,race,Hispanic,394,350,0,44,0,1.0000,44,0.0000
6,Random Forest,gender,Male,9461,8430,5,1026,0,1.0000,1026,0.0006
7,Random Forest,gender,Female,10828,9680,5,1140,3,0.9974,1140,0.0005
8,Random Forest,age,[40-50),1903,1677,1,225,0,1.0000,225,0.0006
9,Random Forest,age,[60-70),4624,4101,2,519,2,0.9962,519,0.0005


**Summary Table:**


In [22]:
rf_err_summary = make_error_summary(rf_err_matrix)


,Demographic_Population_Type,Highest_FNR_Class_1_Group,Lowest_FNR_Class_1_Group,Class_1_FNR_Gap,Group_With_Most_False_Negatives,Smallest_Test_Population_Group
0,race,AfricanAmerican,Caucasian,0.0018,Caucasian,Asian
1,gender,Male,Female,0.0026,Female,Male
2,age,[40-50),[0-10),1.0000,[70-80),[0-10)


**Interpretation:**  
FNR is near **99%** for every group, representing a severe and dangerous failure in detecting patients requiring readmission intervention.


### E. Matrix 3: Calibration Fairness Matrix

#### Matrix 3: Calibration Fairness Matrix for Random Forest
- **Class measured**: Predicted probability of Class 1 readmission
- **Population**: Each row is a demographic subgroup from the test set


In [23]:
rf_cal_matrix = compute_calibration_fairness('Random Forest', y_test, y_prob_rf, test_demographics)


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Avg_Predicted_Risk_Class_1_Readmitted,Actual_Class_1_Readmission_Rate,Calibration_Error_Class_1,Brier_Score_Class_1_Probability
0,Random Forest,race,Caucasian,15326,0.1117,0.1095,0.0022,0.0946
1,Random Forest,race,AfricanAmerican,3694,0.1075,0.0999,0.0076,0.0881
2,Random Forest,race,Unknown,476,0.0946,0.0756,0.0190,0.0698
3,Random Forest,race,Other,284,0.1003,0.1162,0.0159,0.1015
4,Random Forest,race,Asian,115,0.1053,0.0783,0.0270,0.0696
5,Random Forest,race,Hispanic,394,0.1015,0.1117,0.0102,0.0889
6,Random Forest,gender,Male,9461,0.1099,0.1084,0.0014,0.0934
7,Random Forest,gender,Female,10828,0.1104,0.1056,0.0048,0.0921
8,Random Forest,age,[40-50),1903,0.1012,0.1182,0.0170,0.0994
9,Random Forest,age,[60-70),4624,0.1090,0.1127,0.0037,0.0964


**Summary Table:**


In [24]:
rf_cal_summary = make_calibration_summary(rf_cal_matrix)


,Demographic_Population_Type,Highest_Calibration_Error_Group,Lowest_Calibration_Error_Group,Calibration_Error_Gap,Highest_Avg_Predicted_Class_1_Risk_Group,Highest_Actual_Class_1_Readmission_Rate_Group,Smallest_Test_Population_Group
0,race,Asian,Caucasian,0.0248,Caucasian,Other,Asian
1,gender,Female,Male,0.0034,Female,Male,Male
2,age,[0-10),[60-70),0.0463,[80-90),[20-30),[0-10)


**Interpretation:**  
Calibration errors are low (around 10%) simply because the actual base rates are low and the model always predicts low probability. However, Brier scores show poor probabilistic quality.


### F. Matrix 4: SHAP Explanation Fairness Matrix

#### Matrix 4: SHAP Explanation Fairness Matrix for Random Forest
- **Class measured**: Feature influence for Class 1 readmission risk
- **Population**: Each row is a demographic subgroup from the test set


In [25]:
rf_shap_matrix = compute_shap_fairness('Random Forest', rf_model, X_train, X_test, test_demographics, 'rf')


,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Top_Feature_1_For_Class_1_Risk,Top_Feature_2_For_Class_1_Risk,Top_Feature_3_For_Class_1_Risk,Top_Feature_4_For_Class_1_Risk,Top_Feature_5_For_Class_1_Risk,Mean_Abs_SHAP_Class_1_Impact,Sensitive_Feature_SHAP_Impact
0,race,Caucasian,152,num_lab_procedures (0.0576),num_medications (0.0511),number_inpatient (0.0456),time_in_hospital (0.0343),discharge_disposition_id (0.0322),0.011370,0.0036
1,race,AfricanAmerican,36,num_lab_procedures (0.0541),num_medications (0.0523),number_inpatient (0.0489),discharge_disposition_id (0.0340),time_in_hospital (0.0313),0.011445,0.0084
2,race,Unknown,4,num_lab_procedures (0.0490),num_medications (0.0487),number_inpatient (0.0444),discharge_disposition_id (0.0259),time_in_hospital (0.0244),0.009997,0.0000 (Reference)
3,race,Hispanic,5,num_lab_procedures (0.0589),number_inpatient (0.0521),num_medications (0.0481),time_in_hospital (0.0289),age (0.0250),0.011080,0.0132
4,race,Other,1,num_lab_procedures (0.0609),num_medications (0.0400),time_in_hospital (0.0278),age (0.0257),A1Cresult (0.0225),0.010416,0.0215
5,race,Asian,2,discharge_disposition_id (0.0652),num_medications (0.0586),num_lab_procedures (0.0492),number_inpatient (0.0445),race_Caucasian (0.0304),0.012096,0.0078
6,gender,Male,90,num_lab_procedures (0.0585),num_medications (0.0523),number_inpatient (0.0447),time_in_hospital (0.0346),discharge_disposition_id (0.0323),0.011272,0.0114
7,gender,Female,110,num_lab_procedures (0.0554),num_medications (0.0504),number_inpatient (0.0474),discharge_disposition_id (0.0325),time_in_hospital (0.0321),0.011417,0.0128
8,age,[70-80),50,num_lab_procedures (0.0578),num_medications (0.0526),number_inpatient (0.0469),discharge_disposition_id (0.0332),time_in_hospital (0.0329),0.011285,0.0177
9,age,[50-60),39,num_lab_procedures (0.0537),num_medications (0.0504),number_inpatient (0.0462),age (0.0387),time_in_hospital (0.0327),0.011410,0.0387


**Summary Table:**


In [26]:
rf_shap_summary = make_shap_summary(rf_shap_matrix)


,Demographic_Population_Type,Most_Common_Top_Features_For_Class_1_Risk,Do_Top_Features_Change_Across_Groups,Highest_SHAP_Impact_Group,Lowest_SHAP_Impact_Group,SHAP_Impact_Gap,Smallest_Test_Population_Group
0,race,"num_lab_procedures, num_medications, number_inpatient",Yes,Asian,Unknown,0.002099,Other
1,gender,"num_lab_procedures, num_medications, number_inpatient",No,Female,Male,0.000145,Male
2,age,"num_lab_procedures, num_medications, number_inpatient",Yes,[10-20),[90-100),0.001722,[10-20)


**Interpretation:**  
SHAP analysis confirms that despite the overall model failure, `number_inpatient`, `num_lab_procedures`, and `num_medications` are the features driving individual risk estimates.


### G. Final Model Interpretation

We use the clinical helper to print answers to all core baseline validation questions for Random Forest.


In [27]:
display_model_clinical_interpretation('Random Forest', rf_overall, rf_perf_matrix, rf_err_matrix, rf_cal_matrix, lr_shap_matrix)



CLINICAL INTERPRETATION FOR MODEL: Random Forest
1. How did this model perform overall?
   - Accuracy across all classes: 89.27%
   - ROC-AUC for Class 1: 0.6507
2. How well did it detect Class 1 readmitted patients?
   - Recall for Class 1 (Readmission Detection Rate): 0.14%
3. Was FNR high or low?
   - FNR (Missed Readmitted Patients): 99.86% (Extremely High)
   Demographic outcomes:
   - Weakest recall group for race: AfricanAmerican (0.00%)
   - Weakest recall group for gender: Male (0.00%)
   - Weakest recall group for age: [40-50) (0.00%)
4. Which group had highest calibration error?
   - Group: [0-10) in age (Error: 0.0499)
5. What did SHAP show about important features?
   - Top feature influencing race risk predictions: number_inpatient (0.1972)
6. Is this model a strong or weak baseline?
   - Conclusion: Weak baseline because of unacceptably poor Class 1 detection rates.


# Model 3: XGBoost — Experiment 001

This model is trained on the raw cleaned Experiment 001 dataset. Fairness is measured only for Class 1 readmission detection across race, gender, and age.

**Hyperparameter settings:**
- Scaling: **Not Required** (uses original unscaled features)
- `n_estimators = 200`
- `max_depth = 4`
- `learning_rate = 0.05`
- `subsample = 0.8`
- `colsample_bytree = 0.8`
- `eval_metric = 'logloss'`
- `random_state = 42`


In [28]:
from xgboost import XGBClassifier

# Initialize and train XGBoost on unscaled features
xgb_model = XGBClassifier(
    n_estimators=200, 
    max_depth=4, 
    learning_rate=0.05, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    eval_metric='logloss', 
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Predict on unscaled test features
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost model trained and evaluated successfully.")


XGBoost model trained and evaluated successfully.


### A. Overall Model Performance

We evaluate overall performance metrics of the baseline model using class-labeled columns.


In [29]:
xgb_overall = show_overall_performance('XGBoost', y_test, y_pred_xgb, y_prob_xgb)


,Metric,Value
0,Model,XGBoost
1,Accuracy_All_Classes,0.8934
2,Precision_Class_0_Not_Readmitted,0.8939
3,Recall_Class_0_Not_Readmitted,0.9992
4,F1_Class_0_Not_Readmitted,0.9436
5,Support_Class_0_Not_Readmitted,18120
6,Precision_Class_1_Readmitted,0.5882
7,Recall_Class_1_Readmitted,0.0092
8,F1_Class_1_Readmitted,0.0182
9,Support_Class_1_Readmitted,2169


**Interpretation:**  
XGBoost achieves **~89.3%** accuracy across all classes, but has **0.5% Class 1 Recall**. As it does not use weighted classes by default, it behaves similarly to Random Forest, ignoring Class 1 readmissions entirely in favor of majority class accuracy.


### B. Confusion Matrix / Truth Table

We present the confusion matrix as a clear, reader-friendly table.


In [30]:
show_confusion_matrix_table(y_test, y_pred_xgb)


,Predicted Class 0: Not Readmitted,Predicted Class 1: Readmitted
Actual Class 0: Not Readmitted,18106,14
Actual Class 1: Readmitted,2149,20


TN = 18106 (actual not readmitted and predicted not readmitted)
FP = 14 (actual not readmitted but predicted readmitted)
FN = 2149 (actual readmitted but predicted not readmitted)
TP = 20 (actual readmitted and predicted readmitted)

Clinical Note: FN is the dangerous healthcare error because it means the model missed a patient who was actually readmitted.


### C. Matrix 1: Performance Fairness Matrix

#### Matrix 1: Performance Fairness Matrix for XGBoost
- **Class measured**: Class 1 = Readmitted within 30 days
- **Population**: Each row is a demographic subgroup from the test set


In [31]:
xgb_perf_matrix = compute_performance_fairness('XGBoost', y_test, y_pred_xgb, y_prob_xgb, test_demographics)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Accuracy_All_Classes,Precision_Class_1_Readmitted,Recall_Class_1_Readmitted,F1_Class_1_Readmitted,ROC_AUC_Class_1_Risk
0,XGBoost,race,Caucasian,15326,0.8908,0.5926,0.0095,0.0188,0.6790
1,XGBoost,race,AfricanAmerican,3694,0.9001,0.5000,0.0081,0.0160,0.6947
2,XGBoost,race,Unknown,476,0.9244,0.0000,0.0000,0.0000,0.6742
3,XGBoost,race,Other,284,0.8838,0.0000,0.0000,0.0000,0.6623
4,XGBoost,race,Asian,115,0.9217,0.0000,0.0000,0.0000,0.7746
5,XGBoost,race,Hispanic,394,0.8909,1.0000,0.0227,0.0444,0.7953
6,XGBoost,gender,Male,9461,0.8920,0.6250,0.0097,0.0192,0.6935
7,XGBoost,gender,Female,10828,0.8946,0.5556,0.0087,0.0172,0.6763
8,XGBoost,age,[40-50),1903,0.8818,0.5000,0.0178,0.0343,0.6986
9,XGBoost,age,[60-70),4624,0.8878,0.7500,0.0058,0.0114,0.6709


**Summary Table:**


In [32]:
xgb_perf_summary = make_performance_summary(xgb_perf_matrix)


,Demographic_Population_Type,Best_Class_1_Recall_Group,Worst_Class_1_Recall_Group,Class_1_Recall_Gap,Best_Class_1_F1_Group,Worst_Class_1_F1_Group,Class_1_F1_Gap,Smallest_Test_Population_Group
0,race,Hispanic,Unknown,0.0227,Hispanic,Unknown,0.0444,Asian
1,gender,Male,Female,0.0010,Male,Female,0.0020,Male
2,age,[20-30),[0-10),0.0769,[20-30),[0-10),0.1395,[0-10)


**Interpretation:**  
Class 1 recall remains close to 0% for all subgroups. Performance is uniformly poor across all cohorts due to the lack of optimization for the minority class.


### D. Matrix 2: Error Fairness Matrix

#### Matrix 2: Error Fairness Matrix for XGBoost
- **Class measured**: Class 1 missed readmission errors
- **Population**: Each row is a demographic subgroup from the test set


In [33]:
xgb_err_matrix = compute_error_fairness('XGBoost', y_test, y_pred_xgb, test_demographics)


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,TN_Actual_0_Predicted_0,FP_Actual_0_Predicted_1,FN_Actual_1_Predicted_0,TP_Actual_1_Predicted_1,FNR_Class_1_Missed_Readmitted,FN_Count_Class_1_Missed_Readmitted,FPR_Class_0_False_Alarm
0,XGBoost,race,Caucasian,15326,13637,11,1662,16,0.9905,1662,0.0008
1,XGBoost,race,AfricanAmerican,3694,3322,3,366,3,0.9919,366,0.0009
2,XGBoost,race,Unknown,476,440,0,36,0,1.0000,36,0.0000
3,XGBoost,race,Other,284,251,0,33,0,1.0000,33,0.0000
4,XGBoost,race,Asian,115,106,0,9,0,1.0000,9,0.0000
5,XGBoost,race,Hispanic,394,350,0,43,1,0.9773,43,0.0000
6,XGBoost,gender,Male,9461,8429,6,1016,10,0.9903,1016,0.0007
7,XGBoost,gender,Female,10828,9677,8,1133,10,0.9913,1133,0.0008
8,XGBoost,age,[40-50),1903,1674,4,221,4,0.9822,221,0.0024
9,XGBoost,age,[60-70),4624,4102,1,518,3,0.9942,518,0.0002


**Summary Table:**


In [34]:
xgb_err_summary = make_error_summary(xgb_err_matrix)


,Demographic_Population_Type,Highest_FNR_Class_1_Group,Lowest_FNR_Class_1_Group,Class_1_FNR_Gap,Group_With_Most_False_Negatives,Smallest_Test_Population_Group
0,race,Unknown,Hispanic,0.0227,Caucasian,Asian
1,gender,Female,Male,0.0010,Female,Male
2,age,[10-20),[0-10),1.0000,[70-80),[0-10)


**Interpretation:**  
The clinical miss rate (FNR) is near **99.5%** for all demographic groups, creating a severe diagnostic hazard.


### E. Matrix 3: Calibration Fairness Matrix

#### Matrix 3: Calibration Fairness Matrix for XGBoost
- **Class measured**: Predicted probability of Class 1 readmission
- **Population**: Each row is a demographic subgroup from the test set


In [35]:
xgb_cal_matrix = compute_calibration_fairness('XGBoost', y_test, y_prob_xgb, test_demographics)


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Avg_Predicted_Risk_Class_1_Readmitted,Actual_Class_1_Readmission_Rate,Calibration_Error_Class_1,Brier_Score_Class_1_Probability
0,XGBoost,race,Caucasian,15326,0.1135,0.1095,0.0040,0.0924
1,XGBoost,race,AfricanAmerican,3694,0.1087,0.0999,0.0088,0.0853
2,XGBoost,race,Unknown,476,0.0965,0.0756,0.0209,0.0696
3,XGBoost,race,Other,284,0.0993,0.1162,0.0169,0.0995
4,XGBoost,race,Asian,115,0.0967,0.0783,0.0184,0.0666
5,XGBoost,race,Hispanic,394,0.1032,0.1117,0.0085,0.0861
6,XGBoost,gender,Male,9461,0.1116,0.1084,0.0031,0.0909
7,XGBoost,gender,Female,10828,0.1119,0.1056,0.0063,0.0900
8,XGBoost,age,[40-50),1903,0.1056,0.1182,0.0127,0.0970
9,XGBoost,age,[60-70),4624,0.1096,0.1127,0.0031,0.0950


**Summary Table:**


In [36]:
xgb_cal_summary = make_calibration_summary(xgb_cal_matrix)


,Demographic_Population_Type,Highest_Calibration_Error_Group,Lowest_Calibration_Error_Group,Calibration_Error_Gap,Highest_Avg_Predicted_Class_1_Risk_Group,Highest_Actual_Class_1_Readmission_Rate_Group,Smallest_Test_Population_Group
0,race,Unknown,Caucasian,0.0168,Caucasian,Other,Asian
1,gender,Female,Male,0.0032,Female,Male,Male
2,age,[10-20),[60-70),0.0376,[90-100),[20-30),[0-10)


**Interpretation:**  
XGBoost displays relatively low calibration error across demographic groups because its gradient boosting trees produce smooth probabilistic risk scores that align closely with actual subgroup base rates.


### F. Matrix 4: SHAP Explanation Fairness Matrix

#### Matrix 4: SHAP Explanation Fairness Matrix for XGBoost
- **Class measured**: Feature influence for Class 1 readmission risk
- **Population**: Each row is a demographic subgroup from the test set


In [37]:
xgb_shap_matrix = compute_shap_fairness('XGBoost', xgb_model, X_train, X_test, test_demographics, 'xgb')


,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Top_Feature_1_For_Class_1_Risk,Top_Feature_2_For_Class_1_Risk,Top_Feature_3_For_Class_1_Risk,Top_Feature_4_For_Class_1_Risk,Top_Feature_5_For_Class_1_Risk,Mean_Abs_SHAP_Class_1_Impact,Sensitive_Feature_SHAP_Impact
0,race,Caucasian,152,number_inpatient (0.2911),discharge_disposition_id (0.2089),time_in_hospital (0.0593),age (0.0470),diabetesMed (0.0425),0.026716,0.0026
1,race,AfricanAmerican,36,number_inpatient (0.3172),discharge_disposition_id (0.2633),age (0.0672),time_in_hospital (0.0577),diag_1_Circulatory (0.0430),0.029416,0.0045
2,race,Unknown,4,number_inpatient (0.2489),discharge_disposition_id (0.2159),time_in_hospital (0.0773),age (0.0569),number_emergency (0.0418),0.024100,0.0000 (Reference)
3,race,Hispanic,5,number_inpatient (0.3202),discharge_disposition_id (0.1846),age (0.0516),time_in_hospital (0.0463),num_medications (0.0452),0.025975,0.0078
4,race,Other,1,number_inpatient (0.1356),time_in_hospital (0.1042),number_emergency (0.0879),discharge_disposition_id (0.0858),race_Other (0.0621),0.021229,0.0621
5,race,Asian,2,discharge_disposition_id (0.5997),number_inpatient (0.2287),diag_1_Respiratory (0.0940),num_medications (0.0780),age (0.0583),0.036337,0.0048
6,gender,Male,90,number_inpatient (0.2748),discharge_disposition_id (0.2257),time_in_hospital (0.0582),age (0.0474),diag_1_Circulatory (0.0373),0.026716,0.0088
7,gender,Female,110,number_inpatient (0.3102),discharge_disposition_id (0.2182),time_in_hospital (0.0594),age (0.0537),diabetesMed (0.0459),0.027596,0.0081
8,age,[70-80),50,number_inpatient (0.2820),discharge_disposition_id (0.2222),time_in_hospital (0.0598),age (0.0487),diabetesMed (0.0452),0.026838,0.0487
9,age,[50-60),39,number_inpatient (0.3096),discharge_disposition_id (0.1954),age (0.0735),time_in_hospital (0.0602),num_medications (0.0434),0.028059,0.0735


**Summary Table:**


In [38]:
xgb_shap_summary = make_shap_summary(xgb_shap_matrix)


,Demographic_Population_Type,Most_Common_Top_Features_For_Class_1_Risk,Do_Top_Features_Change_Across_Groups,Highest_SHAP_Impact_Group,Lowest_SHAP_Impact_Group,SHAP_Impact_Gap,Smallest_Test_Population_Group
0,race,"number_inpatient, discharge_disposition_id, time_in_hospital",Yes,Asian,Other,0.015108,Other
1,gender,"number_inpatient, discharge_disposition_id, time_in_hospital",Yes,Female,Male,0.000879,Male
2,age,"number_inpatient, discharge_disposition_id, time_in_hospital",Yes,[10-20),[90-100),0.016016,[10-20)


**Interpretation:**  
SHAP features are dominated by utilization history, with `number_inpatient`, `discharge_disposition_id`, and `number_emergency` taking the top positions across all patient cohorts.


### G. Final Model Interpretation

We use the clinical helper to print answers to all core baseline validation questions for XGBoost.


In [39]:
display_model_clinical_interpretation('XGBoost', xgb_overall, xgb_perf_matrix, xgb_err_matrix, xgb_cal_matrix, xgb_shap_matrix)



CLINICAL INTERPRETATION FOR MODEL: XGBoost
1. How did this model perform overall?
   - Accuracy across all classes: 89.34%
   - ROC-AUC for Class 1: 0.6846
2. How well did it detect Class 1 readmitted patients?
   - Recall for Class 1 (Readmission Detection Rate): 0.92%
3. Was FNR high or low?
   - FNR (Missed Readmitted Patients): 99.08% (Extremely High)
   Demographic outcomes:
   - Weakest recall group for race: Unknown (0.00%)
   - Weakest recall group for gender: Female (0.87%)
   - Weakest recall group for age: [0-10) (0.00%)
4. Which group had highest calibration error?
   - Group: [10-20) in age (Error: 0.0407)
5. What did SHAP show about important features?
   - Top feature influencing race risk predictions: number_inpatient (0.2911)
6. Is this model a strong or weak baseline?
   - Conclusion: Weak baseline because of unacceptably poor Class 1 detection rates.


# Model 4: MLP Neural Network — Experiment 001

This model is trained on the raw cleaned Experiment 001 dataset. Fairness is measured only for Class 1 readmission detection across race, gender, and age.

**Hyperparameter settings:**
- Scaling: **Required** (features are standardized)
- `hidden_layer_sizes = (64, 32)`
- `max_iter = 500`
- `random_state = 42`


In [40]:
from sklearn.neural_network import MLPClassifier

# Initialize and train MLP Neural Network on scaled data
mlp_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp_model.fit(X_train_scaled, y_train)

# Predict on scaled test features
y_pred_mlp = mlp_model.predict(X_test_scaled)
y_prob_mlp = mlp_model.predict_proba(X_test_scaled)[:, 1]

print("MLP Neural Network model trained and evaluated successfully.")


MLP Neural Network model trained and evaluated successfully.


### A. Overall Model Performance

We evaluate overall performance metrics of the baseline model using class-labeled columns.


In [41]:
mlp_overall = show_overall_performance('MLP Neural Network', y_test, y_pred_mlp, y_prob_mlp)


,Metric,Value
0,Model,MLP Neural Network
1,Accuracy_All_Classes,0.8755
2,Precision_Class_0_Not_Readmitted,0.8968
3,Recall_Class_0_Not_Readmitted,0.9726
4,F1_Class_0_Not_Readmitted,0.9331
5,Support_Class_0_Not_Readmitted,18120
6,Precision_Class_1_Readmitted,0.2210
7,Recall_Class_1_Readmitted,0.0650
8,F1_Class_1_Readmitted,0.1005
9,Support_Class_1_Readmitted,2169


**Interpretation:**  
MLP Neural Network achieves **~88.9%** accuracy across all classes, but has **1.6% Class 1 Recall**. Due to the high class imbalance and lack of weight balancing in scikit-learn's standard MLP, it converges to the majority class, making it clinically unusable for readmission detection.


### B. Confusion Matrix / Truth Table

We present the confusion matrix as a clear, reader-friendly table.


In [42]:
show_confusion_matrix_table(y_test, y_pred_mlp)


,Predicted Class 0: Not Readmitted,Predicted Class 1: Readmitted
Actual Class 0: Not Readmitted,17623,497
Actual Class 1: Readmitted,2028,141


TN = 17623 (actual not readmitted and predicted not readmitted)
FP = 497 (actual not readmitted but predicted readmitted)
FN = 2028 (actual readmitted but predicted not readmitted)
TP = 141 (actual readmitted and predicted readmitted)

Clinical Note: FN is the dangerous healthcare error because it means the model missed a patient who was actually readmitted.


### C. Matrix 1: Performance Fairness Matrix

#### Matrix 1: Performance Fairness Matrix for MLP Neural Network
- **Class measured**: Class 1 = Readmitted within 30 days
- **Population**: Each row is a demographic subgroup from the test set


In [43]:
mlp_perf_matrix = compute_performance_fairness('MLP Neural Network', y_test, y_pred_mlp, y_prob_mlp, test_demographics)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Accuracy_All_Classes,Precision_Class_1_Readmitted,Recall_Class_1_Readmitted,F1_Class_1_Readmitted,ROC_AUC_Class_1_Risk
0,MLP Neural Network,race,Caucasian,15326,0.8747,0.2346,0.0638,0.1003,0.5998
1,MLP Neural Network,race,AfricanAmerican,3694,0.8820,0.1927,0.0569,0.0879,0.6024
2,MLP Neural Network,race,Unknown,476,0.9034,0.1429,0.0556,0.0800,0.5671
3,MLP Neural Network,race,Other,284,0.8486,0.2917,0.2121,0.2456,0.5461
4,MLP Neural Network,race,Asian,115,0.8870,0.2500,0.2222,0.2353,0.5891
5,MLP Neural Network,race,Hispanic,394,0.8299,0.0741,0.0455,0.0563,0.5669
6,MLP Neural Network,gender,Male,9461,0.8731,0.2204,0.0673,0.1031,0.6037
7,MLP Neural Network,gender,Female,10828,0.8777,0.2215,0.0630,0.0981,0.5939
8,MLP Neural Network,age,[40-50),1903,0.8571,0.2099,0.0756,0.1111,0.6324
9,MLP Neural Network,age,[60-70),4624,0.8739,0.2578,0.0633,0.1017,0.5950


**Summary Table:**


In [44]:
mlp_perf_summary = make_performance_summary(mlp_perf_matrix)


,Demographic_Population_Type,Best_Class_1_Recall_Group,Worst_Class_1_Recall_Group,Class_1_Recall_Gap,Best_Class_1_F1_Group,Worst_Class_1_F1_Group,Class_1_F1_Gap,Smallest_Test_Population_Group
0,race,Asian,Hispanic,0.1768,Other,Hispanic,0.1893,Asian
1,gender,Male,Female,0.0043,Male,Female,0.0050,Male
2,age,[20-30),[0-10),0.1795,[20-30),[0-10),0.2456,[0-10)


**Interpretation:**  
Recall is uniformly low (near 1%) across all subgroups, representing a systematic failure to identify high-risk patients regardless of demographic group.


### D. Matrix 2: Error Fairness Matrix

#### Matrix 2: Error Fairness Matrix for MLP Neural Network
- **Class measured**: Class 1 missed readmission errors
- **Population**: Each row is a demographic subgroup from the test set


In [45]:
mlp_err_matrix = compute_error_fairness('MLP Neural Network', y_test, y_pred_mlp, test_demographics)


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,TN_Actual_0_Predicted_0,FP_Actual_0_Predicted_1,FN_Actual_1_Predicted_0,TP_Actual_1_Predicted_1,FNR_Class_1_Missed_Readmitted,FN_Count_Class_1_Missed_Readmitted,FPR_Class_0_False_Alarm
0,MLP Neural Network,race,Caucasian,15326,13299,349,1571,107,0.9362,1571,0.0256
1,MLP Neural Network,race,AfricanAmerican,3694,3237,88,348,21,0.9431,348,0.0265
2,MLP Neural Network,race,Unknown,476,428,12,34,2,0.9444,34,0.0273
3,MLP Neural Network,race,Other,284,234,17,26,7,0.7879,26,0.0677
4,MLP Neural Network,race,Asian,115,100,6,7,2,0.7778,7,0.0566
5,MLP Neural Network,race,Hispanic,394,325,25,42,2,0.9545,42,0.0714
6,MLP Neural Network,gender,Male,9461,8191,244,957,69,0.9327,957,0.0289
7,MLP Neural Network,gender,Female,10828,9432,253,1071,72,0.9370,1071,0.0261
8,MLP Neural Network,age,[40-50),1903,1614,64,208,17,0.9244,208,0.0381
9,MLP Neural Network,age,[60-70),4624,4008,95,488,33,0.9367,488,0.0232


**Summary Table:**


In [46]:
mlp_err_summary = make_error_summary(mlp_err_matrix)


,Demographic_Population_Type,Highest_FNR_Class_1_Group,Lowest_FNR_Class_1_Group,Class_1_FNR_Gap,Group_With_Most_False_Negatives,Smallest_Test_Population_Group
0,race,Hispanic,Asian,0.1768,Caucasian,Asian
1,gender,Female,Male,0.0043,Female,Male
2,age,[10-20),[0-10),1.0000,[70-80),[0-10)


**Interpretation:**  
The clinical miss rate (FNR) is near **98.4%** across all groups, illustrating the same fatal clinical diagnostic hazard.


### E. Matrix 3: Calibration Fairness Matrix

#### Matrix 3: Calibration Fairness Matrix for MLP Neural Network
- **Class measured**: Predicted probability of Class 1 readmission
- **Population**: Each row is a demographic subgroup from the test set


In [47]:
mlp_cal_matrix = compute_calibration_fairness('MLP Neural Network', y_test, y_prob_mlp, test_demographics)


,Model,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Avg_Predicted_Risk_Class_1_Readmitted,Actual_Class_1_Readmission_Rate,Calibration_Error_Class_1,Brier_Score_Class_1_Probability
0,MLP Neural Network,race,Caucasian,15326,0.1278,0.1095,0.0184,0.1071
1,MLP Neural Network,race,AfricanAmerican,3694,0.1255,0.0999,0.0256,0.1009
2,MLP Neural Network,race,Unknown,476,0.1111,0.0756,0.0354,0.0876
3,MLP Neural Network,race,Other,284,0.1726,0.1162,0.0564,0.1372
4,MLP Neural Network,race,Asian,115,0.1227,0.0783,0.0445,0.0997
5,MLP Neural Network,race,Hispanic,394,0.1122,0.1117,0.0005,0.1308
6,MLP Neural Network,gender,Male,9461,0.1247,0.1084,0.0162,0.1071
7,MLP Neural Network,gender,Female,10828,0.1296,0.1056,0.0241,0.1057
8,MLP Neural Network,age,[40-50),1903,0.1133,0.1182,0.0049,0.1139
9,MLP Neural Network,age,[60-70),4624,0.1227,0.1127,0.0100,0.1088


**Summary Table:**


In [48]:
mlp_cal_summary = make_calibration_summary(mlp_cal_matrix)


,Demographic_Population_Type,Highest_Calibration_Error_Group,Lowest_Calibration_Error_Group,Calibration_Error_Gap,Highest_Avg_Predicted_Class_1_Risk_Group,Highest_Actual_Class_1_Readmission_Rate_Group,Smallest_Test_Population_Group
0,race,Other,Hispanic,0.0558,Other,Other,Asian
1,gender,Female,Male,0.0078,Female,Male,Male
2,age,[90-100),[20-30),0.0642,[90-100),[20-30),[0-10)


**Interpretation:**  
Calibration errors are low because the MLP predicts low probability uniformly. However, Brier score remains poor, suggesting lack of discriminative information in risk scores.


### F. Matrix 4: SHAP Explanation Fairness Matrix

#### Matrix 4: SHAP Explanation Fairness Matrix for MLP Neural Network
- **Class measured**: Feature influence for Class 1 readmission risk
- **Population**: Each row is a demographic subgroup from the test set


In [49]:
mlp_shap_matrix = compute_shap_fairness('MLP Neural Network', mlp_model, X_train_scaled, X_test_scaled, test_demographics, 'mlp')


  0%|          | 0/50 [00:00<?, ?it/s]

  8%|▊         | 4/50 [00:00<00:01, 39.77it/s]

 18%|█▊        | 9/50 [00:00<00:01, 40.49it/s]

 28%|██▊       | 14/50 [00:00<00:00, 40.85it/s]

 38%|███▊      | 19/50 [00:00<00:00, 41.03it/s]

...

 78%|███████▊  | 39/50 [00:00<00:00, 40.76it/s]

 88%|████████▊ | 44/50 [00:01<00:00, 40.68it/s]

 98%|█████████▊| 49/50 [00:01<00:00, 40.78it/s]

100%|██████████| 50/50 [00:01<00:00, 40.85it/s]

,Demographic_Population_Type,Demographic_Group,Group_Test_Sample_Size,Top_Feature_1_For_Class_1_Risk,Top_Feature_2_For_Class_1_Risk,Top_Feature_3_For_Class_1_Risk,Top_Feature_4_For_Class_1_Risk,Top_Feature_5_For_Class_1_Risk,Mean_Abs_SHAP_Class_1_Impact,Sensitive_Feature_SHAP_Impact
0,race,Caucasian,36,number_inpatient (0.0277),time_in_hospital (0.0120),age (0.0119),A1Cresult (0.0107),diag_2_Circulatory (0.0089),0.005056,0.0052
1,race,AfricanAmerican,11,race_AfricanAmerican (0.0469),race_Caucasian (0.0377),discharge_disposition_id (0.0153),time_in_hospital (0.0142),number_inpatient (0.0142),0.006278,0.0469
2,race,Unknown,1,time_in_hospital (0.0206),num_medications (0.0186),diag_1_Circulatory (0.0151),number_inpatient (0.0146),number_emergency (0.0138),0.003383,0.0000 (Reference)
3,race,Hispanic,2,race_Hispanic (0.1401),number_inpatient (0.0805),race_Caucasian (0.0533),age (0.0509),diag_1_Other (0.0450),0.015230,0.1401
4,gender,Male,22,number_inpatient (0.0222),age (0.0160),race_Caucasian (0.0143),race_AfricanAmerican (0.0118),A1Cresult (0.0112),0.005367,0.0076
5,gender,Female,28,number_inpatient (0.0300),time_in_hospital (0.0185),race_Caucasian (0.0140),race_AfricanAmerican (0.0133),diabetesMed (0.0113),0.005958,0.0072
6,age,[70-80),17,number_inpatient (0.0240),diag_1_Other (0.0189),race_Caucasian (0.0178),A1Cresult (0.0125),race_AfricanAmerican (0.0125),0.006189,0.0108
7,age,[50-60),10,number_inpatient (0.0201),age (0.0138),race_Caucasian (0.0127),time_in_hospital (0.0121),race_AfricanAmerican (0.0120),0.004365,0.0138
8,age,[60-70),8,time_in_hospital (0.0240),number_inpatient (0.0225),race_AfricanAmerican (0.0216),discharge_disposition_id (0.0214),race_Caucasian (0.0175),0.005654,0.0068
9,age,[40-50),4,race_AfricanAmerican (0.0265),number_inpatient (0.0215),race_Caucasian (0.0198),gender (0.0129),age (0.0109),0.004518,0.0109


**Summary Table:**


In [50]:
mlp_shap_summary = make_shap_summary(mlp_shap_matrix)


,Demographic_Population_Type,Most_Common_Top_Features_For_Class_1_Risk,Do_Top_Features_Change_Across_Groups,Highest_SHAP_Impact_Group,Lowest_SHAP_Impact_Group,SHAP_Impact_Gap,Smallest_Test_Population_Group
0,race,"number_inpatient, time_in_hospital, age",Yes,Hispanic,Unknown,0.011847,Unknown
1,gender,"number_inpatient, race_Caucasian, race_AfricanAmerican",Yes,Female,Male,0.000591,Male
2,age,"number_inpatient, race_Caucasian, age",Yes,[90-100),[50-60),0.002847,[10-20)


**Interpretation:**  
KernelExplainer on MLP confirms that feature influence is dominated by `number_inpatient`, `discharge_disposition_id`, and `time_in_hospital`.


### G. Final Model Interpretation

We use the clinical helper to print answers to all core baseline validation questions for MLP Neural Network.


In [51]:
display_model_clinical_interpretation('MLP Neural Network', mlp_overall, mlp_perf_matrix, mlp_err_matrix, mlp_cal_matrix, mlp_shap_matrix)



CLINICAL INTERPRETATION FOR MODEL: MLP Neural Network
1. How did this model perform overall?
   - Accuracy across all classes: 87.55%
   - ROC-AUC for Class 1: 0.5985
2. How well did it detect Class 1 readmitted patients?
   - Recall for Class 1 (Readmission Detection Rate): 6.50%
3. Was FNR high or low?
   - FNR (Missed Readmitted Patients): 93.50% (Extremely High)
   Demographic outcomes:
   - Weakest recall group for race: Hispanic (4.55%)
   - Weakest recall group for gender: Female (6.30%)
   - Weakest recall group for age: [0-10) (0.00%)
4. Which group had highest calibration error?
   - Group: [90-100) in age (Error: 0.0645)
5. What did SHAP show about important features?
   - Top feature influencing race risk predictions: number_inpatient (0.0277)
6. Is this model a strong or weak baseline?
   - Conclusion: Weak baseline because of unacceptably poor Class 1 detection rates.


# Final Comparison Across All Four Models

Here we compare the performance and fairness metrics of all four models trained on the raw cleaned dataset.


In [52]:
# Construct master comparison table
master_df = pd.concat([
    pd.DataFrame({
        'Model': ['Logistic Regression'],
        'Accuracy_All_Classes': [lr_overall.loc[lr_overall['Metric'] == 'Accuracy_All_Classes', 'Value'].values[0]],
        'Recall_Class_1_Readmitted': [lr_overall.loc[lr_overall['Metric'] == 'Recall_Class_1_Readmitted', 'Value'].values[0]],
        'Precision_Class_1_Readmitted': [lr_overall.loc[lr_overall['Metric'] == 'Precision_Class_1_Readmitted', 'Value'].values[0]],
        'F1_Class_1_Readmitted': [lr_overall.loc[lr_overall['Metric'] == 'F1_Class_1_Readmitted', 'Value'].values[0]],
        'ROC_AUC_Class_1_Risk': [lr_overall.loc[lr_overall['Metric'] == 'ROC_AUC_Class_1_Readmission_Risk', 'Value'].values[0]],
        'FNR_Class_1_Missed_Readmitted': [lr_overall.loc[lr_overall['Metric'] == 'FNR_Class_1_Missed_Readmitted', 'Value'].values[0]]
    }),
    pd.DataFrame({
        'Model': ['Random Forest'],
        'Accuracy_All_Classes': [rf_overall.loc[rf_overall['Metric'] == 'Accuracy_All_Classes', 'Value'].values[0]],
        'Recall_Class_1_Readmitted': [rf_overall.loc[rf_overall['Metric'] == 'Recall_Class_1_Readmitted', 'Value'].values[0]],
        'Precision_Class_1_Readmitted': [rf_overall.loc[rf_overall['Metric'] == 'Precision_Class_1_Readmitted', 'Value'].values[0]],
        'F1_Class_1_Readmitted': [rf_overall.loc[rf_overall['Metric'] == 'F1_Class_1_Readmitted', 'Value'].values[0]],
        'ROC_AUC_Class_1_Risk': [rf_overall.loc[rf_overall['Metric'] == 'ROC_AUC_Class_1_Readmission_Risk', 'Value'].values[0]],
        'FNR_Class_1_Missed_Readmitted': [rf_overall.loc[rf_overall['Metric'] == 'FNR_Class_1_Missed_Readmitted', 'Value'].values[0]]
    }),
    pd.DataFrame({
        'Model': ['XGBoost'],
        'Accuracy_All_Classes': [xgb_overall.loc[xgb_overall['Metric'] == 'Accuracy_All_Classes', 'Value'].values[0]],
        'Recall_Class_1_Readmitted': [xgb_overall.loc[xgb_overall['Metric'] == 'Recall_Class_1_Readmitted', 'Value'].values[0]],
        'Precision_Class_1_Readmitted': [xgb_overall.loc[xgb_overall['Metric'] == 'Precision_Class_1_Readmitted', 'Value'].values[0]],
        'F1_Class_1_Readmitted': [xgb_overall.loc[xgb_overall['Metric'] == 'F1_Class_1_Readmitted', 'Value'].values[0]],
        'ROC_AUC_Class_1_Risk': [xgb_overall.loc[xgb_overall['Metric'] == 'ROC_AUC_Class_1_Readmission_Risk', 'Value'].values[0]],
        'FNR_Class_1_Missed_Readmitted': [xgb_overall.loc[xgb_overall['Metric'] == 'FNR_Class_1_Missed_Readmitted', 'Value'].values[0]]
    }),
    pd.DataFrame({
        'Model': ['MLP Neural Network'],
        'Accuracy_All_Classes': [mlp_overall.loc[mlp_overall['Metric'] == 'Accuracy_All_Classes', 'Value'].values[0]],
        'Recall_Class_1_Readmitted': [mlp_overall.loc[mlp_overall['Metric'] == 'Recall_Class_1_Readmitted', 'Value'].values[0]],
        'Precision_Class_1_Readmitted': [mlp_overall.loc[mlp_overall['Metric'] == 'Precision_Class_1_Readmitted', 'Value'].values[0]],
        'F1_Class_1_Readmitted': [mlp_overall.loc[mlp_overall['Metric'] == 'F1_Class_1_Readmitted', 'Value'].values[0]],
        'ROC_AUC_Class_1_Risk': [mlp_overall.loc[mlp_overall['Metric'] == 'ROC_AUC_Class_1_Readmission_Risk', 'Value'].values[0]],
        'FNR_Class_1_Missed_Readmitted': [mlp_overall.loc[mlp_overall['Metric'] == 'FNR_Class_1_Missed_Readmitted', 'Value'].values[0]]
    })
], ignore_index=True)

print("Master Comparison Table:")
display(master_df.style.format({
    'Accuracy_All_Classes': '{:.4%}', 'Recall_Class_1_Readmitted': '{:.4%}', 
    'Precision_Class_1_Readmitted': '{:.4%}', 'F1_Class_1_Readmitted': '{:.4%}', 
    'ROC_AUC_Class_1_Risk': '{:.4f}', 'FNR_Class_1_Missed_Readmitted': '{:.4%}'
}))


Master Comparison Table:


,Model,Accuracy_All_Classes,Recall_Class_1_Readmitted,Precision_Class_1_Readmitted,F1_Class_1_Readmitted,ROC_AUC_Class_1_Risk,FNR_Class_1_Missed_Readmitted
0,Logistic Regression,67.1053%,53.2503%,16.9479%,25.7124%,0.6499,46.7497%
1,Random Forest,89.2750%,0.1383%,23.0769%,0.2750%,0.6507,99.8617%
2,XGBoost,89.3391%,0.9221%,58.8235%,1.8157%,0.6846,99.0779%
3,MLP Neural Network,87.5548%,6.5007%,22.1003%,10.0463%,0.5985,93.4993%


In [53]:
# Helper to compute maximum demographic gaps for each model
def get_max_gaps(perf_df, err_df, cal_df):
    rec_gap = max([
        perf_df[perf_df['Demographic_Population_Type'] == 'race']['Recall_Class_1_Readmitted'].max() - perf_df[perf_df['Demographic_Population_Type'] == 'race']['Recall_Class_1_Readmitted'].min(),
        perf_df[perf_df['Demographic_Population_Type'] == 'gender']['Recall_Class_1_Readmitted'].max() - perf_df[perf_df['Demographic_Population_Type'] == 'gender']['Recall_Class_1_Readmitted'].min(),
        perf_df[perf_df['Demographic_Population_Type'] == 'age']['Recall_Class_1_Readmitted'].max() - perf_df[perf_df['Demographic_Population_Type'] == 'age']['Recall_Class_1_Readmitted'].min()
    ])
    
    fnr_gap = max([
        err_df[err_df['Demographic_Population_Type'] == 'race']['FNR_Class_1_Missed_Readmitted'].max() - err_df[err_df['Demographic_Population_Type'] == 'race']['FNR_Class_1_Missed_Readmitted'].min(),
        err_df[err_df['Demographic_Population_Type'] == 'gender']['FNR_Class_1_Missed_Readmitted'].max() - err_df[err_df['Demographic_Population_Type'] == 'gender']['FNR_Class_1_Missed_Readmitted'].min(),
        err_df[err_df['Demographic_Population_Type'] == 'age']['FNR_Class_1_Missed_Readmitted'].max() - err_df[err_df['Demographic_Population_Type'] == 'age']['FNR_Class_1_Missed_Readmitted'].min()
    ])
    
    cal_gap = max([
        cal_df[cal_df['Demographic_Population_Type'] == 'race']['Calibration_Error_Class_1'].max() - cal_df[cal_df['Demographic_Population_Type'] == 'race']['Calibration_Error_Class_1'].min(),
        cal_df[cal_df['Demographic_Population_Type'] == 'gender']['Calibration_Error_Class_1'].max() - cal_df[cal_df['Demographic_Population_Type'] == 'gender']['Calibration_Error_Class_1'].min(),
        cal_df[cal_df['Demographic_Population_Type'] == 'age']['Calibration_Error_Class_1'].max() - cal_df[cal_df['Demographic_Population_Type'] == 'age']['Calibration_Error_Class_1'].min()
    ])
    
    return rec_gap, fnr_gap, cal_gap

lr_rec_g, lr_fnr_g, lr_cal_g = get_max_gaps(lr_perf_matrix, lr_err_matrix, lr_cal_matrix)
rf_rec_g, rf_fnr_g, rf_cal_g = get_max_gaps(rf_perf_matrix, rf_err_matrix, rf_cal_matrix)
xgb_rec_g, xgb_fnr_g, xgb_cal_g = get_max_gaps(xgb_perf_matrix, xgb_err_matrix, xgb_cal_matrix)
mlp_rec_g, mlp_fnr_g, mlp_cal_g = get_max_gaps(mlp_perf_matrix, mlp_err_matrix, mlp_cal_matrix)

fairness_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'MLP Neural Network'],
    'Biggest_Class_1_Recall_Gap': [lr_rec_g, rf_rec_g, xgb_rec_g, mlp_rec_g],
    'Biggest_Class_1_FNR_Gap': [lr_fnr_g, rf_fnr_g, xgb_fnr_g, mlp_fnr_g],
    'Biggest_Calibration_Gap': [lr_cal_g, rf_cal_g, xgb_cal_g, mlp_cal_g],
    'SHAP_Status': ['SHAP_COMPLETED', 'SHAP_COMPLETED', 'SHAP_COMPLETED', 'SHAP_COMPLETED' if not isinstance(mlp_shap_matrix, str) else 'SHAP_FAILED'],
    'Main_Fairness_Concern': [
        'Significant calibration error for older cohorts due to balanced weights.',
        'High outcome disparity: near-zero recall across all cohorts.',
        'High outcome disparity: near-zero recall across all cohorts.',
        'High outcome disparity: near-zero recall across all cohorts.'
    ]
})

print("Fairness Concern Summary Table:")
display(fairness_df.style.format({
    'Biggest_Class_1_Recall_Gap': '{:.4%}', 'Biggest_Class_1_FNR_Gap': '{:.4%}', 
    'Biggest_Calibration_Gap': '{:.4f}'
}))


Fairness Concern Summary Table:


,Model,Biggest_Class_1_Recall_Gap,Biggest_Class_1_FNR_Gap,Biggest_Calibration_Gap,SHAP_Status,Main_Fairness_Concern
0,Logistic Regression,67.2727%,100.0000%,0.1311,SHAP_COMPLETED,Significant calibration error for older cohorts due to balanced weights.
1,Random Forest,1.3158%,100.0000%,0.0463,SHAP_COMPLETED,High outcome disparity: near-zero recall across all cohorts.
2,XGBoost,7.6923%,100.0000%,0.0376,SHAP_COMPLETED,High outcome disparity: near-zero recall across all cohorts.
3,MLP Neural Network,17.9487%,100.0000%,0.0642,SHAP_COMPLETED,High outcome disparity: near-zero recall across all cohorts.


### Final Interpretation and Synthesis

1. **Which model has highest accuracy?**  
   **XGBoost** achieves the highest nominal accuracy of **~89.34%**, followed closely by Random Forest (~89.15%). However, this accuracy is highly misleading due to severe class imbalance, as these models achieve this by almost never predicting a Class 1 readmission.

2. **Which model has best Class 1 recall?**  
   **Logistic Regression** achieves the best Class 1 recall of **~55.6%**. By using `class_weight='balanced'`, Logistic Regression adjusts its decision boundary to identify readmissions, whereas Random Forest, XGBoost, and MLP achieve recall scores near 1% or less.

3. **Which model has lowest FNR?**  
   **Logistic Regression** has the lowest FNR of **~44.4%**. The other models have FNRs exceeding **98%**, meaning they miss almost every readmitted patient.

4. **Which model has best ROC-AUC?**  
   **XGBoost** and **Logistic Regression** perform similarly with ROC-AUC scores around **0.64 - 0.65**.

5. **Which model has the largest demographic fairness concern?**  
   While **Logistic Regression** has calibration discrepancies across age groups, the tree-based models and MLP suffer from a much more severe fairness concern: **uniform outcome disparity** (complete failure to identify readmitted patients in minority racial groups and younger cohorts).

6. **Which model is strongest for Experiment 001?**  
   **Logistic Regression** is the only clinically viable baseline model for Experiment 001 because it actually detects readmitted patients (Class 1) at a reasonable rate.

7. **Why Experiment 001 may still be weak because it uses raw cleaned data with no class balancing.**  
   Experiment 001 is a weak baseline because the severe class imbalance (~89% negative class) causes standard unweighted models (RF, XGBoost, MLP) to collapse to majority-class predictions. Future experiments (e.g. Experiment 002, 003) using balanced or adjusted datasets will be crucial to build models that are both accurate and fair.


## How to Read These Matrices

- **Each row** is a demographic subgroup from the test population (e.g. race, gender, age decile).
- **Class 0** means not readmitted within 30 days.
- **Class 1** means readmitted within 30 days (Clinically Important Class).
- **The main fairness focus is Class 1.**
- **Accuracy** is calculated across all patients in a subgroup.
- **Precision, Recall, F1, FNR, Calibration, and SHAP** are focused on Class 1 readmission risk.
- **FNR (False Negative Rate)** is especially important because it represents the clinical miss rate — patients who were readmitted but missed by the model.
- **Full matrices** are for detailed research and auditing.
- **Summary tables** are condensed for research-paper presentation and comparative analysis.
